# OWSM + sEEG (B2T'25) — versión Google Colab con barrido de experimentos

Adaptación del notebook local para Colab. Cambios principales respecto a la versión local:

## Novedades de esta versión (v3)

Tras la ablación de la capa por sesión (`sinSesion` 0.952 vs `conSesion` 0.989 de `cer_ctc`, ambos con `wer=1.000`): los dos brazos están al nivel del azar, así que la comparación no dice nada. Tres arreglos:

| Problema | Arreglo |
|---|---|
| `best_model_criterion` era `valid loss`, que con `ctc_weight=0.3` es 70 % pérdida de atención y toca fondo en la época 4 | Nuevo `criterio="cer_ctc"`: se guarda el checkpoint con mejor `cer_ctc` |
| Se reportaba el `cer_ctc` *de la época de menor loss*, no el mejor | `min_valid_cer_ctc` + `epoch_min_cer_ctc` + `pendiente_cer_ctc` (¿sigue bajando o está plano?) |
| `cer_ctc_greedy` no era comparable con la columna del `train.log` | Ahora se reportan las dos: `cer_ctc_micro` (idéntica a la de ESPnet) y `cer_texto` |

## Cómo usarlo

1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU L4**. Marca *High RAM* solo si vas a usar `n_sessions >= 30` con `PRELOAD_RAM = True`.
2. Comprueba que `DRIVE_DATA` (celda 1) apunta a tu carpeta `hdf5_data_final` en Drive.
3. Ejecuta las celdas 0 a 10 en orden. Reinicia el entorno cuando te lo pida la celda de instalación.
4. Edita la lista `BARRIDO` (celda 11) y lanza la celda 12.

Los resultados se van acumulando en `resultados.csv` en tu Drive: si Colab te desconecta, vuelves a lanzar y se salta los experimentos ya hechos.

## 0. Instalación

Ejecuta esta celda, **reinicia el entorno** (`Entorno de ejecución → Reiniciar sesión`) y continúa desde la celda 1. Si no reinicias, numpy se queda cargado en la versión antigua y aparecen errores de tipos difíciles de diagnosticar.

In [4]:
!nvidia-smi

# Instalación (~3 min la primera vez).
# Truco para sesiones siguientes: descarga las ruedas a Drive una sola vez con
#   !pip download -q espnet espnet_model_zoo loralib h5py -d /content/drive/MyDrive/TFG/wheels
# y luego instala offline en ~20 s con
#   !pip install -q --no-index --find-links=/content/drive/MyDrive/TFG/wheels espnet espnet_model_zoo loralib h5py
!pip install -q espnet espnet_model_zoo loralib h5py

import torch
print("PyTorch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("SIN GPU: revisa Entorno de ejecucion > Cambiar tipo de entorno de ejecucion")
print("\n>>> REINICIA EL ENTORNO AHORA y sigue desde la celda 1 <<<")

Wed Jul 29 14:51:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             16W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Panel de control

Todo lo configurable vive aquí. `DEFAULTS` son los valores base de un experimento; en el barrido (celda 11) solo indicas lo que cambia respecto a estos.

In [5]:
# ══ RUTAS ══════════════════════════════════════════════════════════
DRIVE_ROOT  = "/content/drive/MyDrive/TFG"                       # <<< tu carpeta en Drive
DRIVE_DATA  = f"{DRIVE_ROOT}/DatosEEG_crudos/hdf5_data_final"    # <<< donde estan los HDF5
DATA_ROOT   = "/content/data/hdf5_data_final"    # cache en el disco local de la VM
OUTPUT_DIR  = "/content/exp"                     # checkpoints en local, NO en Drive
RESULTS_DIR = f"{DRIVE_ROOT}/resultados"         # solo lo que quieres conservar

# ══ MODELO ═════════════════════════════════════════════════════════
FINETUNE_MODEL = "espnet/owsm_v3.1_ebf_base"       # ~101M params
LANGUAGE       = "eng"
LORA_TARGET    = ["w_1", "w_2", "merge_proj", "linear_q", "linear_k", "linear_v", "linear_out"]

# ══ DATOS ══════════════════════════════════════════════════════════
PRELOAD_RAM   = False   # False = lectura perezosa del HDF5 (seguro con 45 sesiones)
                        # True  = todo en RAM (~1.6 MB/trial; necesita High RAM si n_sessions>25)
SOLO_SESIONES_COMPLETAS = True   # ignora sesiones sin data_val.hdf5 (p.ej. t15.2023.08.11)

# ══ EVALUACION ═════════════════════════════════════════════════════
EVAL_N    = 50    # trials de validacion sobre los que calcular CER/WER al final de cada run
BEAM_SIZE = 5
EVAL_GREEDY_CTC = True   # ademas del beam search, decodificacion CTC greedy (metrica honesta:
                         # no pasa por el decoder, asi que no puede "hacer trampa" con el
                         # prior del ingles preentrenado)

# ══ CONTROL DEL BARRIDO ════════════════════════════════════════════
SKIP_IF_DONE = True    # no repetir experimentos ya presentes en resultados.csv
GUARDAR_CKPT = False   # copiar el mejor .pth a Drive (~470 MB por experimento)

# ══ VALORES POR DEFECTO DE UN EXPERIMENTO ══════════════════════════
DEFAULTS = dict(
    nombre        = "base",     # etiqueta legible para las tablas de la memoria
    n_sessions    = 10,         # 45 disponibles
    max_epoch     = 30,
    warmup        = 100,
    val_igual_train = False,    # True = test de sobreajuste (val == train)

    frontend      = "conv2d",   # "conv2d" | "linear" | "conv1d"
    capa_sesion   = False,      # True = transformada afin por sesion + softsign delante del
                                # frontend (es LA caracteristica del baseline de B2T'25: la
                                # senal del mismo fonema cambia de un dia a otro por deriva
                                # de los electrodos, y 45 sesiones abarcan 20 meses)
    norm          = "trial",    # "trial"  = z-score por trial y canal (lo que hacias)
                                # "ninguna"= usar los HDF5 tal cual (ya vienen z-scoreados
                                #            por bloque; los trabajos publicados no renormalizan)
    unfreeze_encoder = True,
    unfreeze_n_layers = None,   # None = encoder entero; int = solo las N primeras capas
    usar_lora     = True,

    batch_type    = "sorted",   # "sorted" (batch_size) | "numel" (batch_bins)
    batch_size    = 16,
    batch_bins    = 6_000_000,
    accum_grad    = 1,

    lr            = 1e-3,
    enc_lr_scale  = 1.0,        # <1.0 = LR menor para el encoder preentrenado (p.ej. 0.01)
    weight_decay  = 1e-6,
    grad_clip     = 5.0,
    ctc_weight    = 0.3,        # ahora SI afecta al loss de entrenamiento
    ctc_weight_decode = None,   # None = mismo que ctc_weight
    seed          = 2024,
    criterio      = "cer_ctc",  # metrica con la que se elige el mejor checkpoint.
                                # "cer_ctc" = la del sEEG. "loss" = la total, que con
                                # ctc_weight bajo es sobre todo perdida de atencion y
                                # toca fondo en 4-5 epocas por el prior del ingles.
    num_workers   = 0,          # 0 = sin fork. Con HDF5 es lo mas seguro y no cuesta
                                # nada: el modelo es compute-bound y los datos estan en
                                # disco local. Sube a 2 solo si ves la GPU esperando.
)

print("Panel cargado.")

Panel cargado.


## 2. Drive y datos

Tus HDF5 ya están en Drive, así que solo hay que montarlo. Lo que **no** conviene es entrenar leyendo directamente de `/content/drive`: Drive limita las operaciones de E/S por fichero y acaba dando `OSError: [Errno 5] Input/output error` a mitad de una época.

La solución es una caché: las sesiones se copian de Drive al disco local de la VM **solo cuando un experimento las necesita**. Si el barrido empieza con 2 sesiones, se copian 2 (unos segundos) y no las 45.

Esta celda solo monta Drive y comprueba qué hay. La copia ocurre sola más adelante.

In [6]:
import os, time, shutil, glob

from google.colab import drive
drive.mount("/content/drive")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

# Localizar la carpeta de datos aunque este en otro sitio dentro de DRIVE_ROOT
if not os.path.isdir(DRIVE_DATA):
    print(f"No esta en {DRIVE_DATA}; buscando 'hdf5_data_final' bajo {DRIVE_ROOT} ...")
    cands = glob.glob(f"{DRIVE_ROOT}/**/hdf5_data_final", recursive=True)
    if cands:
        DRIVE_DATA = cands[0]
        print("Encontrada:", DRIVE_DATA)
    else:
        raise FileNotFoundError(
            f"No encuentro la carpeta hdf5_data_final dentro de {DRIVE_ROOT}.\n"
            f"Ajusta DRIVE_ROOT / DRIVE_DATA en el panel de control."
        )

# Inventario: que sesiones hay y cuales tienen train Y val
inventario, sin_val, bytes_tot = [], [], 0
for s in sorted(os.listdir(DRIVE_DATA)):
    d = os.path.join(DRIVE_DATA, s)
    if not os.path.isdir(d):
        continue
    tr = os.path.join(d, "data_train.hdf5")
    va = os.path.join(d, "data_val.hdf5")
    if not os.path.exists(tr):
        continue
    inventario.append(s)
    if not os.path.exists(va):
        sin_val.append(s)
    bytes_tot += os.path.getsize(tr) + (os.path.getsize(va) if os.path.exists(va) else 0)

print(f"\n{len(inventario)} sesiones con data_train.hdf5 · {bytes_tot/1e9:.1f} GB (train+val)")
if sin_val:
    print(f"{len(sin_val)} sin data_val.hdf5: {sin_val}")
    print("  -> con SOLO_SESIONES_COMPLETAS=True se ignoran (evita que train y val no cuadren)")
print("\nPrimeras sesiones:", inventario[:5])
print("Disco libre en la VM:")
os.system("df -h /content | tail -1")

Mounted at /content/drive

45 sesiones con data_train.hdf5 · 11.2 GB (train+val)
4 sin data_val.hdf5: ['t15.2023.08.11', 't15.2024.03.03', 't15.2024.04.25', 't15.2024.04.28']
  -> con SOLO_SESIONES_COMPLETAS=True se ignoran (evita que train y val no cuadren)

Primeras sesiones: ['t15.2023.08.11', 't15.2023.08.13', 't15.2023.08.18', 't15.2023.08.20', 't15.2023.08.25']
Disco libre en la VM:


0

## 3. Imports, dispositivo y utilidades

In [7]:
import string, re, glob, argparse, logging, gc, copy, json, math
import h5py
import numpy as np
import pandas as pd
import torch
import loralib

import espnetez as ez
from espnet2.bin.s2t_inference import Speech2Text
from espnet2.layers.create_adapter_fn import create_lora_adapter
from espnet.nets.pytorch_backend.transformer.subsampling import Conv2dSubsampling

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
NGPU    = 1 if torch.cuda.is_available() else 0
USE_AMP = torch.cuda.is_available()

GPU_NAME = torch.cuda.get_device_name(0) if NGPU else "CPU"

# Tarifas aproximadas de compute units/hora (medidas en marzo de 2026; pueden variar)
TARIFAS = {"T4": 1.19, "L4": 1.71, "A100-SXM4-40GB": 5.40, "A100-SXM4-80GB": 7.52,
           "RTX PRO 6000": 8.71, "H100": 9.0}
def units_por_hora(nombre_gpu):
    for k, v in TARIFAS.items():
        if k.lower().replace("-", " ") in nombre_gpu.lower().replace("-", " "):
            return v
    return float("nan")
CU_H = units_por_hora(GPU_NAME)

print(f"Dispositivo: {DEVICE} · {GPU_NAME} · AMP: {USE_AMP}")
print(f"Coste estimado: {CU_H} compute units/hora (~${CU_H*0.10:.2f}/h)")


def normaliza(x, modo="trial"):
    """Normalizacion de la senal sEEG antes de entrar al modelo.

    Los HDF5 de B2T'25 ya vienen binneados a 20 ms y z-scoreados por bloque, asi que
    "ninguna" reproduce lo que hacen los trabajos publicados; "trial" vuelve a normalizar
    por trial y canal (util para comparar, pero borra la amplitud relativa entre trials).
    """
    x = x.astype(np.float32)
    if modo == "ninguna":
        return x
    return (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-8)

def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

def decode_transcription(arr):
    arr = np.asarray(arr).ravel()
    return "".join(chr(int(x)) for x in arr if int(x) != 0)

Failed to import Flash Attention, using ESPnet default: No module named 'flash_attn'


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.
/usr/local/lib/python3.12/dist-packages/espnet2/enh/encoder/stft_encoder.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/usr/local/lib/python3.12/dist-packages/espnet2/enh/layers/uses2_swin.py:329: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


Dispositivo: cuda · NVIDIA L4 · AMP: True
Coste estimado: 1.71 compute units/hora (~$0.17/h)


## 4. Datasets (carga perezosa)

`SeeGDataset` guarda solo un **índice** `(fichero, clave)` y lee cada trial del HDF5 cuando hace falta. Así las 45 sesiones (~17 GB en float32) no tienen que caber en RAM. El handle de h5py se reabre por proceso para que funcione con `num_workers > 0`.

Con `PRELOAD_RAM = True` vuelve al comportamiento local (todo en memoria), que es algo más rápido si tienes RAM de sobra.

In [8]:
class SeeGDataset(torch.utils.data.Dataset):
    """Lee trials del HDF5 bajo demanda. Si preload=True los cachea todos en RAM."""

    def __init__(self, index, preload=False):
        self.index = index          # [(ruta_hdf5, clave), ...]
        self.preload = preload
        self._files = {}
        self._pid = os.getpid()
        self._cache = None
        if preload:
            self._cache = [self._leer(i) for i in range(len(index))]
            self.cerrar()

    def _handle(self, path):
        if os.getpid() != self._pid:      # tras un fork, los handles del padre no valen
            self._files, self._pid = {}, os.getpid()
        if path not in self._files:
            self._files[path] = h5py.File(path, "r")
        return self._files[path]

    def cerrar(self):
        """Cierra los handles. IMPRESCINDIBLE antes de que el DataLoader haga fork:
        la libreria HDF5 no es fork-safe y heredar handles abiertos puede colgar el proceso."""
        for f in self._files.values():
            try: f.close()
            except Exception: pass
        self._files = {}

    def _leer(self, idx):
        path, key, sid = self.index[idx]
        t = self._handle(path)[key]
        return {"input_features": t["input_features"][:],
                "text_raw": decode_transcription(t["transcription"][()]),
                "session_idx": sid}

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        d = self._cache[idx] if self._cache is not None else self._leer(idx)
        text_lower = d["text_raw"].lower()
        return {
            "input_features": d["input_features"],
            "session_idx": d["session_idx"],
            "text":      f"<{LANGUAGE}><asr><notimestamps> {text_lower}",
            "text_prev": "<na>",
            "text_ctc":  remove_punctuation(text_lower),
            "text_raw":  d["text_raw"],
        }


def construir_indice(sesiones, split):
    """Recorre las claves de cada HDF5 sin leer los datos.

    Cada entrada es (ruta, clave, indice_de_sesion). El indice es la posicion en la
    lista `sesiones`, la misma para train y val, y es lo que consume la capa por sesion.
    """
    index = []
    for sid, s in enumerate(sesiones):
        path = os.path.join(DATA_ROOT, s, f"data_{split}.hdf5")
        if not os.path.exists(path):
            print(f"  [aviso] no encontrado: {path}")
            continue
        with h5py.File(path, "r") as f:
            index.extend((path, k, sid) for k in sorted(f.keys()))
    return index


def sesiones_disponibles():
    """Sesiones utilizables, listadas desde Drive (la fuente de la verdad)."""
    out = []
    for s in sorted(os.listdir(DRIVE_DATA)):
        d = os.path.join(DRIVE_DATA, s)
        if not os.path.isdir(d) or not os.path.exists(os.path.join(d, "data_train.hdf5")):
            continue
        if SOLO_SESIONES_COMPLETAS and not os.path.exists(os.path.join(d, "data_val.hdf5")):
            continue
        out.append(s)
    return out


def copiar_sesion(sesion):
    """Trae data_train/data_val de esa sesion al disco local de la VM (solo la primera vez)."""
    src = os.path.join(DRIVE_DATA, sesion)
    dst = os.path.join(DATA_ROOT, sesion)
    os.makedirs(dst, exist_ok=True)
    for split in ("train", "val"):
        f_src = os.path.join(src, f"data_{split}.hdf5")
        f_dst = os.path.join(dst, f"data_{split}.hdf5")
        if os.path.exists(f_src) and not os.path.exists(f_dst):
            shutil.copy(f_src, f_dst)      # data_test.hdf5 no se copia: no tiene etiquetas
    return dst


def elegir_sesiones(n_sessions):
    ses = sesiones_disponibles()[:n_sessions]
    faltan = [s for s in ses
              if not os.path.exists(os.path.join(DATA_ROOT, s, "data_train.hdf5"))]
    if faltan:
        print(f"Copiando {len(faltan)} sesion(es) de Drive al disco local de la VM...")
        t0 = time.time()
        for i, s in enumerate(faltan, 1):
            copiar_sesion(s)
            print(f"  [{i}/{len(faltan)}] {s}")
        print(f"  copiadas en {time.time()-t0:.0f}s")
    return ses


_CACHE_DATOS = {}

def get_datos(cfg):
    """Devuelve (train_raw, val_raw, stats_dir, n_sesiones_reales).

    La clave de cache incluye todo lo que cambia la FORMA del tensor de entrada
    (capa_sesion anade un canal: 512 -> 513), porque los shape files de ESPnet
    dejan de valer si eso cambia.
    """
    n_sessions      = cfg["n_sessions"]
    val_igual_train = cfg["val_igual_train"]
    clave = (n_sessions, val_igual_train, PRELOAD_RAM, cfg["capa_sesion"])
    if clave in _CACHE_DATOS:
        return _CACHE_DATOS[clave]

    sesiones = elegir_sesiones(n_sessions)
    print(f"Sesiones ({len(sesiones)}): {sesiones[:3]}{' ...' if len(sesiones) > 3 else ''}")
    idx_train = construir_indice(sesiones, "train")
    idx_val   = idx_train if val_igual_train else construir_indice(sesiones, "val")

    train_raw = SeeGDataset(idx_train, preload=PRELOAD_RAM)
    val_raw   = train_raw if val_igual_train else SeeGDataset(idx_val, preload=PRELOAD_RAM)

    stats_dir = (f"{OUTPUT_DIR}/stats_n{n_sessions}"
                 f"{'_vt' if val_igual_train else ''}"
                 f"{'_sess' if cfg['capa_sesion'] else ''}")
    print(f"Train: {len(idx_train)} trials · Val: {len(idx_val)} trials")

    _CACHE_DATOS[clave] = (train_raw, val_raw, stats_dir, len(sesiones))
    return _CACHE_DATOS[clave]


# Comprobacion rapida: copia UNA sesion y lee un trial
_ses = sesiones_disponibles()[:1]
assert _ses, "No hay sesiones utilizables; revisa DRIVE_DATA y SOLO_SESIONES_COMPLETAS"
copiar_sesion(_ses[0])
_ds = SeeGDataset(construir_indice(_ses, "train"))
_it = _ds[0]
_x  = _it["input_features"]
print(f"\nSesion de prueba: {_ses[0]} ({len(_ds)} trials)")
print("text    :", _it["text"])
print("text_ctc:", _it["text_ctc"])
print("features:", _x.shape, _x.dtype)
# Los HDF5 de B2T'25 ya vienen z-scoreados por bloque: si esto ya esta cerca de (0, 1),
# volver a normalizar por trial (norm="trial") borra la amplitud relativa entre trials.
print(f"estadisticas del HDF5 crudo: media={_x.mean():+.3f} std={_x.std():.3f} "
      f"min={_x.min():+.2f} max={_x.max():+.2f}")


Sesion de prueba: t15.2023.08.13 (348 trials)
text    : <eng><asr><notimestamps> which is most unfortunate because we all lose out.
text_ctc: which is most unfortunate because we all lose out
features: (1023, 512) float32
estadisticas del HDF5 crudo: media=-0.000 std=0.999 min=-2.64 max=+10.00


## 5. Tokenizador y configuración base de OWSM

Se carga el modelo preentrenado **una sola vez** por sesión: de él salen el tokenizador, la config base y una copia de los pesos que reutilizan todos los experimentos.

In [9]:
pretrained = Speech2Text.from_pretrained(FINETUNE_MODEL, lang_sym=f"<{LANGUAGE}>", beam_size=5)
pretrain_config = vars(pretrained.s2t_train_args)
tokenizer = pretrained.tokenizer
converter = pretrained.converter
_pretrained_state_dict = {k: v.cpu().clone() for k, v in pretrained.s2t_model.state_dict().items()}
del pretrained
gc.collect()

def tokenize(text):
    return np.array(converter.tokens2ids(tokenizer.text2tokens(text)), dtype=np.int64)

BLANK_ID = converter.token_list.index("<blank>") if "<blank>" in converter.token_list else 0


def preparar_speech(d, cfg):
    """Tensor de entrada al modelo: (T, 512) o (T, 513) si hay capa por sesion.

    El indice de sesion viaja como un canal extra constante. Es la forma menos invasiva
    de meterlo: el pipeline de ESPnet solo transporta 'speech', y el padding va al final
    (con ceros), asi que el frame 0 siempre es dato real y de ahi lo lee el frontend.
    """
    x = normaliza(d["input_features"], cfg["norm"])
    if cfg["capa_sesion"]:
        sid = np.full((x.shape[0], 1), float(d["session_idx"]), dtype=np.float32)
        x = np.concatenate([x, sid], axis=1)
    return x


def make_data_info(cfg):
    return {
        "speech":    lambda d: preparar_speech(d, cfg),
        "text":      lambda d: tokenize(d["text"]),
        "text_prev": lambda d: tokenize(d["text_prev"]),
        "text_ctc":  lambda d: tokenize(d["text_ctc"]),
    }

print("Tokenizador y config base cargados")
print("  ctc_weight original de OWSM:", pretrain_config.get("model_conf", {}).get("ctc_weight"))
print("  id del blank de CTC:", BLANK_ID, "· vocabulario:", len(converter.token_list))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

Tokenizador y config base cargados
  ctc_weight original de OWSM: 0.3
  id del blank de CTC: 0 · vocabulario: 50002


## 6. Construcción del modelo

Tres arreglos respecto a la versión local:

1. **`ctc_weight` se inyecta en `model_conf`.** Antes el modelo se construía desde `pretrain_config`, así que el peso del CTC en el loss era siempre el 0.3 de OWSM, dijeras lo que dijeras en el panel. (Se veía en los logs: `0.3·loss_ctc + 0.7·loss_att = loss`.)
2. **`usar_lora` es opcional**, para poder comparar con/sin adaptadores.
3. **`enc_lr_scale`** parchea `build_optimizers` para poner el encoder preentrenado en un grupo de parámetros con LR más bajo. Con Adam no vale escalar gradientes (es invariante a escala), hace falta grupos de verdad.

**Nota sobre los frontends `linear`/`conv1d`:** heredan de `Conv2dSubsampling` aunque no usen su `__init__`. Es necesario porque el encoder de ESPnet decide si pasarle la máscara de padding al frontend mirando `isinstance(self.embed, Conv2dSubsampling)`; si la clase no hereda de ahí, ESPnet asume que es un embedding que no cambia la longitud de la secuencia (como un `nn.Embedding` de texto) y lo llama sin máscara, lo que revienta con `TypeError: forward() missing 1 required positional argument: 'x_mask'`.

In [10]:
def count_trainable(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False


def _subsample_mask(x_mask, subsample, target_len):
    """Recorta/rellena la mascara para que case EXACTAMENTE con la longitud de x tras el submuestreo."""
    if x_mask is None:
        return None
    m = x_mask[:, :, ::subsample]
    if m.size(2) > target_len:
        m = m[:, :, :target_len]
    elif m.size(2) < target_len:
        m = torch.nn.functional.pad(m, (0, target_len - m.size(2)), value=False)
    return m


class CapaSesion(torch.nn.Module):
    """Transformada afin por sesion + softsign (capa 'day-specific' del baseline de B2T'25).

    Por que hace falta: los 45 dias de grabacion abarcan 20 meses y la senal que produce
    un mismo fonema cambia de un dia a otro (deriva de los electrodos, micromovimientos
    del array). Un unico frontend compartido tiene que aprender N mapeos senal->fonema
    contradictorios a la vez y ademas adivinar de que dia es cada trial. Con una matriz
    por sesion, cada dia se alinea a un espacio comun y el resto del modelo ve un unico
    problema coherente.

    Se inicializa a la identidad, asi que al empezar solo aplica el softsign.
    Coste: n_sesiones x 512 x 512 parametros (~2.6 M con 10 sesiones, ~12 M con 45).
    """

    def __init__(self, n_sesiones, dim=512):
        super().__init__()
        self.n_sesiones = n_sesiones
        self.W = torch.nn.Parameter(torch.eye(dim).unsqueeze(0).repeat(n_sesiones, 1, 1))
        self.b = torch.nn.Parameter(torch.zeros(n_sesiones, dim))

    def forward(self, x):
        # x: (B, T, dim+1). El ultimo canal es el indice de sesion, constante en el tiempo.
        sid = x[:, 0, -1].round().long().clamp_(0, self.n_sesiones - 1)
        x = x[:, :, :-1]
        # baddbmm: b[sid] + x @ W[sid], por muestra del batch
        x = torch.baddbmm(self.b[sid].unsqueeze(1), x, self.W[sid])
        return torch.nn.functional.softsign(x)


def build_frontend(kind, pos_enc, capa_sesion=None):
    """Frontend (encoder.embed) que reemplaza al Conv2dSubsampling original de mel.

    IMPORTANTE: heredan de Conv2dSubsampling (aunque no usen su __init__) porque el
    encoder de ESPnet decide si pasar la mascara al embed mirando
    `isinstance(self.embed, Conv2dSubsampling)`. Si no se hereda de ahi, ESPnet llama
    a embed(x) SIN mascara (asumiendo que es un embedding de texto que no submuestrea)
    y revienta con "forward() missing 1 required positional argument: 'x_mask'".
    """
    import torch.nn as nn

    if kind == "conv2d":
        class Conv2dFrontend(Conv2dSubsampling):
            # OJO: ~66 GFLOP/muestra, el 85-90% del coste del modelo.
            def __init__(self):
                super().__init__(512, 384, dropout_rate=0.0, pos_enc=pos_enc)
                self.sesion = capa_sesion
            def forward(self, x, x_mask):
                if self.sesion is not None:
                    x = self.sesion(x)
                return super().forward(x, x_mask)
        return Conv2dFrontend()

    elif kind == "linear":
        class LinearFrontend(Conv2dSubsampling):
            def __init__(self, in_dim=512, out_dim=384, subsample=4):
                torch.nn.Module.__init__(self)   # NO llamar a Conv2dSubsampling.__init__
                self.sesion = capa_sesion
                self.proj = nn.Linear(in_dim, out_dim)
                self.subsample = subsample
                self.pos_enc = pos_enc
            def forward(self, x, x_mask):
                if self.sesion is not None:
                    x = self.sesion(x)
                x = self.proj(x)
                x = x[:, ::self.subsample, :]
                x = self.pos_enc(x)
                x_mask = _subsample_mask(x_mask, self.subsample, x.size(1))
                return x, x_mask
        return LinearFrontend()

    elif kind == "conv1d":
        class Conv1dFrontend(Conv2dSubsampling):
            def __init__(self, in_dim=512, out_dim=384):
                torch.nn.Module.__init__(self)   # NO llamar a Conv2dSubsampling.__init__
                self.sesion = capa_sesion
                self.conv = nn.Sequential(
                    nn.Conv1d(in_dim, out_dim, kernel_size=3, stride=2, padding=1), nn.ReLU(),
                    nn.Conv1d(out_dim, out_dim, kernel_size=3, stride=2, padding=1), nn.ReLU(),
                )
                self.pos_enc = pos_enc
            def forward(self, x, x_mask):
                if self.sesion is not None:
                    x = self.sesion(x)
                x = self.conv(x.transpose(1, 2)).transpose(1, 2)
                x = self.pos_enc(x)
                x_mask = _subsample_mask(x_mask, 4, x.size(1))
                return x, x_mask
        return Conv1dFrontend()

    raise ValueError(f"Frontend desconocido: {kind}")


def make_build_model_fn(cfg, n_sesiones=1, verbose=True):
    """Devuelve el build_model_fn que ESPnet-EZ usara para ESTE experimento."""
    def build_model_fn(args):
        from espnet2.tasks.s2t import S2TTask

        # ── ARREGLO: el ctc_weight del panel tiene que llegar al modelo ──
        conf = dict(pretrain_config)
        conf["model_conf"] = {**conf.get("model_conf", {}), "ctc_weight": cfg["ctc_weight"]}

        model = S2TTask.build_model(argparse.Namespace(**conf))
        model.load_state_dict(_pretrained_state_dict, strict=False)

        # sEEG entra directa: fuera frontend mel y normalizacion global
        model.frontend = None
        model.normalize = None

        pos_enc = model.encoder.embed.out[1]
        sesion = CapaSesion(n_sesiones) if cfg["capa_sesion"] else None
        model.encoder.embed = build_frontend(cfg["frontend"], pos_enc, sesion)

        model.train()
        freeze_all(model)
        if cfg["usar_lora"]:
            create_lora_adapter(model, target_modules=LORA_TARGET)

        for p in model.encoder.embed.parameters():   # frontend nuevo (incl. capa de sesion)
            p.requires_grad = True
        if model.ctc is not None:                    # ESPnet lo pone a None si ctc_weight==0
            for p in model.ctc.parameters():
                p.requires_grad = True

        if cfg["unfreeze_encoder"]:
            if cfg["unfreeze_n_layers"] is None:
                for p in model.encoder.parameters():
                    p.requires_grad = True
            else:
                for layer in model.encoder.encoders[:cfg["unfreeze_n_layers"]]:
                    for p in layer.parameters():
                        p.requires_grad = True

        if verbose:
            total, trainable = count_trainable(model)
            extra = f" + capa_sesion({n_sesiones})" if cfg["capa_sesion"] else ""
            print(f"  frontend={cfg['frontend']}{extra} · lora={cfg['usar_lora']} · "
                  f"encoder={'descongelado' if cfg['unfreeze_encoder'] else 'congelado'} · "
                  f"ctc_weight={cfg['ctc_weight']} · norm={cfg['norm']}")
            print(f"  {trainable:,} entrenables / {total:,} totales ({trainable/total*100:.2f}%)")
            if model.ctc is None:
                print("  [nota] ctc_weight=0 -> ESPnet desactiva la cabeza CTC")
            if getattr(model, "decoder", None) is None:
                print("  [nota] ctc_weight=1 -> ESPnet desactiva el decoder "
                      "(solo CTC; la evaluacion sera greedy)")
        return model

    return build_model_fn


# ── LR diferencial: grupos de parametros en el optimizador ────────────
def set_encoder_lr_scale(scale):
    """scale<1.0 -> el encoder preentrenado entrena con lr*scale; el frontend nuevo, CTC y LoRA con lr."""
    from espnet2.tasks.s2t import S2TTask

    if scale is None or scale == 1.0:
        if "build_optimizers" in S2TTask.__dict__:
            del S2TTask.build_optimizers      # restaura la implementacion original
        return

    OPTIMS = {"adam": torch.optim.Adam, "adamw": torch.optim.AdamW, "sgd": torch.optim.SGD}

    def build_optimizers(cls, args, model):
        if args.optim not in OPTIMS:
            raise ValueError(f"enc_lr_scale no soportado con optim={args.optim}")
        base_lr = args.optim_conf.get("lr", 1e-3)
        conf = {k: v for k, v in args.optim_conf.items() if k != "lr"}
        nuevos, preentrenados = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            es_nuevo = (name.startswith("encoder.embed") or name.startswith("ctc.")
                        or "lora_" in name)
            (nuevos if es_nuevo else preentrenados).append(p)
        grupos = [{"params": nuevos, "lr": base_lr},
                  {"params": preentrenados, "lr": base_lr * scale}]
        print(f"  LR diferencial: {len(nuevos)} tensores a {base_lr:g} · "
              f"{len(preentrenados)} a {base_lr*scale:g}")
        return [OPTIMS[args.optim](grupos, lr=base_lr, **conf)]

    S2TTask.build_optimizers = classmethod(build_optimizers)

print("Constructores de modelo listos.")

Constructores de modelo listos.


## 7. Configuración de entrenamiento

Además de lo que ya inyectabas en local, aquí van los ajustes específicos de GPU: `use_amp`, `cudnn_benchmark`, `cudnn_deterministic=False` (el valor por defecto `True` cuesta un 20-30 %), `keep_nbest_models=1` para no llenar el disco y `resume=True` para sobrevivir a las desconexiones de Colab.

In [11]:
YAML_BASE = """
use_lora: true

rir_scp: null
noise_scp: null
speech_volume_normalize: null
non_linguistic_symbols: null

preprocessor_conf:
  speech_name: speech
  text_name: text

seed: 2024
num_workers: 0
ngpu: 1
batch_type: sorted
batch_size: 8
accum_grad: 1
max_epoch: 1
patience: null
init: null
best_model_criterion:
- [valid, loss, min]
keep_nbest_models: 1
use_amp: true

optim: adam
optim_conf:
    lr: 0.001
    weight_decay: 0.000001
scheduler: warmuplr
scheduler_conf:
    warmup_steps: 20

specaug: null
ctc_weight: 0.0
grad_clip: 5.0
"""

with open("/content/finetune_b2t25.yaml", "w") as f:
    f.write(YAML_BASE)


def build_finetune_config(cfg, stats_dir):
    ft = ez.config.update_finetune_config(
        "s2t", copy.deepcopy(pretrain_config), "/content/finetune_b2t25.yaml")

    # ── entorno ──
    ft["ngpu"] = NGPU
    ft["use_amp"] = USE_AMP
    ft["num_workers"] = cfg["num_workers"]
    # cudnn_benchmark=True es CONTRAPRODUCENTE aqui: con batch_type sorted cada batch
    # tiene una longitud distinta, y cudnn vuelve a buscar el mejor algoritmo para cada
    # forma nueva (143 busquedas exhaustivas en la primera epoca).
    ft["cudnn_benchmark"] = False
    ft["cudnn_deterministic"] = False
    ft["multiple_iterator"] = False
    ft["iterator_type"] = "sequence"
    ft["log_interval"] = 10
    ft["num_iters_per_epoch"] = None
    ft["seed"] = cfg["seed"]
    ft["resume"] = True
    ft["keep_nbest_models"] = 1

    # ── criterio para elegir el mejor checkpoint ──
    # ARREGLO: con ctc_weight bajo, 'valid loss' es sobre todo perdida de atencion y
    # toca fondo hacia la epoca 4, cuando el cer_ctc todavia esta bajando. Seleccionar
    # por cer_ctc guarda el modelo que de verdad es mejor en la tarea.
    if cfg["criterio"] == "cer_ctc" and cfg["ctc_weight"] > 0:
        ft["best_model_criterion"] = [["valid", "cer_ctc", "min"]]
    else:
        ft["best_model_criterion"] = [["valid", "loss", "min"]]

    # ── batching ──
    ft["batch_type"] = cfg["batch_type"]
    if cfg["batch_type"] == "numel":
        ft["batch_bins"] = cfg["batch_bins"]
    else:
        ft["batch_size"] = cfg["batch_size"]
    ft["accum_grad"] = cfg["accum_grad"]

    # ── optimizacion ──
    ft["max_epoch"] = cfg["max_epoch"]
    ft["scheduler_conf"]["warmup_steps"] = cfg["warmup"]
    ft["optim_conf"]["lr"] = cfg["lr"]
    ft["optim_conf"]["weight_decay"] = cfg["weight_decay"]
    ft["grad_clip"] = cfg["grad_clip"]
    ft["ctc_weight"] = cfg["ctc_weight"]          # informativo, el que manda es model_conf
    ft["model_conf"] = {**ft.get("model_conf", {}), "ctc_weight": cfg["ctc_weight"]}

    # ── shape files ──
    nombres = ["speech_shape", "text_shape", "text_prev_shape", "text_ctc_shape"]
    ft["train_shape_file"] = [f"{stats_dir}/train/{n}" for n in nombres]
    ft["valid_shape_file"] = [f"{stats_dir}/valid/{n}" for n in nombres]
    return ft


def shape_files(stats_dir):
    nombres = ["speech_shape", "text_shape", "text_prev_shape", "text_ctc_shape"]
    return ([f"{stats_dir}/train/{n}" for n in nombres]
            + [f"{stats_dir}/valid/{n}" for n in nombres])

print("Constructor de config listo.")

Constructor de config listo.


## 8. Logs y métricas por época

ESPnet no escribe `train.log` por sí solo cuando se lanza desde un notebook (en las recetas sale de una redirección de shell). Aquí se captura el logger raíz a un fichero por experimento y solo se muestran en pantalla las líneas de resumen de época, así el notebook no se llena de miles de líneas.

`parse_train_log` convierte ese log en un CSV con una fila por época: justo lo que necesitas para las curvas de la memoria.

In [12]:
class CapturaLog:
    """Redirige el logging de ESPnet a un fichero; en pantalla, solo el resumen por epoca."""

    CLAVES = ("batch:", "epoch results", "epoch started", "Saving", "best",
              "There are no improvements", "Stop training", "The training was finished")

    def __init__(self, path):
        self.path = path
        self.previos = None

    def __enter__(self):
        root = logging.getLogger()
        self.previos = (root.handlers[:], root.level)
        root.handlers = []
        root.setLevel(logging.INFO)

        fh = logging.FileHandler(self.path, mode="a", encoding="utf-8")
        fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s: %(message)s"))
        root.addHandler(fh)

        claves = self.CLAVES
        class Resumen(logging.Filter):
            def filter(self, record):
                return any(c in record.getMessage() for c in claves)
        sh = logging.StreamHandler()
        sh.addFilter(Resumen())
        sh.setFormatter(logging.Formatter("    %(message)s"))
        root.addHandler(sh)
        return self

    def __exit__(self, *exc):
        root = logging.getLogger()
        for h in root.handlers:
            try: h.close()
            except Exception: pass
        root.handlers, root.level = self.previos
        return False


PAR = re.compile(r"([a-zA-Z_][a-zA-Z_0-9]*)=(-?[\d.]+(?:[eE][-+]?\d+)?)")

def parse_train_log(path):
    """Extrae una fila por epoca del train.log de ESPnet."""
    filas = []
    if not os.path.exists(path):
        return pd.DataFrame()
    with open(path, encoding="utf-8", errors="ignore") as f:
        for linea in f:
            m = re.search(r"(\d+)epoch results:(.*)", linea)
            if not m:
                continue
            epoca, resto = int(m.group(1)), m.group(2)
            fila = {"epoch": epoca}
            partes = re.split(r"\[valid\]", resto)
            for prefijo, trozo in zip(["train_", "valid_"], partes):
                trozo = trozo.replace("[train]", "")
                for k, v in PAR.findall(trozo):
                    if k in ("time", "total_count"):
                        continue
                    fila[prefijo + k] = float(v)
            filas.append(fila)
    df = pd.DataFrame(filas)
    if len(df):
        df = df.drop_duplicates(subset="epoch", keep="last").sort_values("epoch")
    return df

print("Utilidades de log listas.")

Utilidades de log listas.


## 9. Inferencia y CER/WER

Igual que en local (beam search de OWSM reconectado al modelo fine-tuneado), pero el objeto `Speech2Text` se construye **una sola vez** y se reutiliza en todos los experimentos, y los pesos de decodificación (`ctc_weight`) se ajustan sobre la marcha.

In [13]:
_S2T = None

def get_s2t():
    global _S2T
    if _S2T is None:
        _S2T = Speech2Text.from_pretrained(
            FINETUNE_MODEL, lang_sym=f"<{LANGUAGE}>", task_sym="<asr>",
            beam_size=BEAM_SIZE, ctc_weight=0.3, device=DEVICE, nbest=1)
    return _S2T


def rebind_beam_search(s2t, model, ctc_weight):
    """Repunta decoder y CTC del beam search hacia el modelo fine-tuneado."""
    s2t.s2t_model = model
    bs = s2t.beam_search
    for d in (getattr(bs, "scorers", {}), getattr(bs, "full_scorers", {}),
              getattr(bs, "part_scorers", {})):
        if "decoder" in d:
            d["decoder"] = model.decoder
        if "ctc" in d and hasattr(d["ctc"], "ctc"):
            d["ctc"].ctc = model.ctc
    if hasattr(bs, "nn_dict") and "decoder" in bs.nn_dict:
        bs.nn_dict["decoder"] = model.decoder
    if hasattr(bs, "weights") and ctc_weight is not None:
        bs.weights["ctc"] = ctc_weight
        bs.weights["decoder"] = 1.0 - ctc_weight
    bs.to(device=DEVICE).eval()


def _norm(t):
    t = t.lower().replace("\x00", "")
    t = re.sub(r"<[^>]+>", " ", t)
    t = t.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", t).strip()

def _lev(a, b):
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            old = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (ca != cb))
            prev = old
    return dp[-1]

def cer(ref, hyp):
    r, h = _norm(ref), _norm(hyp)
    return 0.0 if len(r) == 0 else _lev(list(r), list(h)) / len(r)

def wer(ref, hyp):
    r, h = _norm(ref).split(), _norm(hyp).split()
    return 0.0 if len(r) == 0 else _lev(r, h) / len(r)


def find_best_checkpoint(exp_dir):
    for name in ["valid.cer_ctc.best.pth", "valid.loss.best.pth",
                 "valid.acc.best.pth", "train.loss.best.pth"]:
        p = os.path.join(exp_dir, name)
        if os.path.exists(p):
            return p
    cands = sorted(glob.glob(os.path.join(exp_dir, "*epoch.pth")), key=os.path.getmtime)
    if not cands:
        raise FileNotFoundError(f"No hay checkpoints en {exp_dir}")
    return cands[-1]


def cargar_modelo_entrenado(cfg, exp_dir, n_sesiones):
    ckpt = find_best_checkpoint(exp_dir)
    modelo = make_build_model_fn(cfg, n_sesiones, verbose=False)(None).to(DEVICE)
    try:
        state = torch.load(ckpt, map_location=DEVICE, weights_only=True)
    except Exception:
        state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    faltan = modelo.load_state_dict(state, strict=False)
    if faltan.missing_keys:
        print(f"    [aviso] {len(faltan.missing_keys)} tensores sin cargar del checkpoint, "
              f"p.ej. {faltan.missing_keys[0]}")
    return modelo.eval()


@torch.no_grad()
def evaluar_ctc_greedy(cfg, modelo, val_raw, n=EVAL_N, mostrar=3):
    """Decodificacion CTC greedy: argmax por frame, colapsar repeticiones, quitar blanks.
    No pasa por el decoder, asi que mide SOLO lo que el encoder extrae del sEEG.

    Devuelve DOS cifras, porque no son la misma:
      - cer_ctc_micro: exactamente como lo calcula ESPnet en el train.log (cadena de
        tokens BPE concatenados, sum(errores)/sum(longitudes)). Es la unica comparable
        con la columna cer_ctc de las curvas.
      - cer_texto: sobre el texto detokenizado y normalizado, media por trial. Es la
        que se parece a un CER "de verdad", pero la hunden los trials cortos que fallan.
    """
    from itertools import groupby
    n = min(n, len(val_raw))
    idxs = np.linspace(0, len(val_raw) - 1, n).astype(int)   # repartidos, no los n primeros

    err_bpe = ref_bpe = 0
    cers_txt, wers_txt, ejemplos = [], [], []
    for i in idxs:
        d = val_raw[int(i)]
        x = preparar_speech(d, cfg)
        speech = torch.tensor(x, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        lens = torch.tensor([speech.shape[1]], dtype=torch.long, device=DEVICE)
        enc, _ = modelo.encode(speech, lens)
        if isinstance(enc, tuple):
            enc = enc[0]
        ids = modelo.ctc.ctc_lo(enc).argmax(-1)[0].tolist()
        hyp_ids = [k for k, _ in groupby(ids) if k != BLANK_ID]   # colapso CTC estandar
        ref_ids = tokenize(d["text_ctc"]).tolist()

        h = "".join(converter.ids2tokens(hyp_ids))
        r = "".join(converter.ids2tokens(ref_ids))
        err_bpe += _lev(list(h), list(r)); ref_bpe += len(r)

        hip = tokenizer.tokens2text(converter.ids2tokens(hyp_ids)) if hyp_ids else ""
        cers_txt.append(cer(d["text_raw"], hip))
        wers_txt.append(wer(d["text_raw"], hip))
        if len(ejemplos) < mostrar:
            ejemplos.append((_norm(d["text_raw"]), hip))

    for ref, hip in ejemplos:
        print(f"    [ctc] REF: {ref}\n    [ctc] HYP: {hip}")
    c = np.array(cers_txt)
    print(f"    por trial: mediana={np.median(c):.3f} p10={np.percentile(c,10):.3f} "
          f"p90={np.percentile(c,90):.3f} · {(c<0.2).mean()*100:.0f}% por debajo de 0.2")
    return err_bpe / max(1, ref_bpe), float(c.mean()), float(np.mean(wers_txt))


@torch.no_grad()
def evaluar(cfg, modelo, val_raw, n=EVAL_N, mostrar=3):
    """CER/WER con el beam search de OWSM (decoder + CTC)."""
    w_dec = cfg["ctc_weight_decode"]
    w_dec = cfg["ctc_weight"] if w_dec is None else w_dec
    s2t = get_s2t()
    rebind_beam_search(s2t, modelo, w_dec)

    lang_id   = s2t.converter.token2id[f"<{LANGUAGE}>"]
    task_id   = s2t.converter.token2id["<asr>"]
    notime_id = s2t.converter.token2id[s2t.preprocessor_conf["notime_symbol"]]

    nivel = logging.getLogger().level
    logging.getLogger().setLevel(logging.WARNING)   # el beam search es MUY verboso
    cers, wers, ejemplos = [], [], []
    try:
        n = min(n, len(val_raw))
        for i in range(n):
            d = val_raw[i]
            x = preparar_speech(d, cfg)
            speech = torch.tensor(x, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            lens = torch.tensor([speech.shape[1]], dtype=torch.long, device=DEVICE)
            enc, _ = modelo.encode(speech, lens)
            if isinstance(enc, tuple):
                enc = enc[0]
            s2t.beam_search.set_hyp_primer([modelo.sos, lang_id, task_id, notime_id])
            res = s2t._decode_single_sample(enc[0])
            text, token, token_int, text_nospecial, hyp = res[0]
            hipotesis = (text_nospecial or text or "").strip()
            cers.append(cer(d["text_raw"], hipotesis))
            wers.append(wer(d["text_raw"], hipotesis))
            if i < mostrar:
                ejemplos.append((_norm(d["text_raw"]), hipotesis))
    finally:
        logging.getLogger().setLevel(nivel)

    for ref, hip in ejemplos:
        print(f"    REF: {ref}\n    HYP: {hip}")
    # Un modelo que ignora la senal sEEG produce siempre la misma frase generica:
    if ejemplos and len({h for _, h in ejemplos}) == 1:
        print("    [aviso] hipotesis identicas: el modelo esta ignorando la entrada")

    return float(np.mean(cers)), float(np.mean(wers)), n

print("Evaluacion lista.")

Evaluacion lista.


## 10. `run_experiment`

Une todo: datos → stats (cacheadas) → entrenamiento → métricas → guardado. Cada llamada deja en Drive una fila en `resultados.csv`, el `train.log` y un `epocas.csv` con la curva completa.

In [14]:
CSV_RESULTADOS = f"{RESULTS_DIR}/resultados.csv"

def hacer_tag(cfg):
    return (f"{cfg['nombre']}_n{cfg['n_sessions']}_{cfg['frontend']}"
            f"{'_sess' if cfg['capa_sesion'] else ''}"
            f"_unf{int(cfg['unfreeze_encoder'])}_lora{int(cfg['usar_lora'])}"
            f"_lr{cfg['lr']:g}_els{cfg['enc_lr_scale']:g}_ctc{cfg['ctc_weight']:g}"
            f"_bs{cfg['batch_size']}_ep{cfg['max_epoch']}")

def experimentos_hechos():
    if not os.path.exists(CSV_RESULTADOS):
        return set()
    try:
        return set(pd.read_csv(CSV_RESULTADOS)["exp_tag"].astype(str))
    except Exception:
        return set()

def guardar_resultado(fila):
    df_nueva = pd.DataFrame([fila])
    if os.path.exists(CSV_RESULTADOS):
        df = pd.concat([pd.read_csv(CSV_RESULTADOS), df_nueva], ignore_index=True)
    else:
        df = df_nueva
    df.to_csv(CSV_RESULTADOS, index=False)


def asegurar_stats(trainer, stats_dir):
    faltan = [p for p in shape_files(stats_dir)
              if not os.path.exists(p) or os.path.getsize(p) == 0]
    if not faltan:
        print("  shape files ya existen, se salta collect_stats")
        return
    print("  recopilando estadisticas (solo la primera vez por configuracion de datos)...")
    nivel = logging.getLogger().level
    logging.getLogger().setLevel(logging.WARNING)
    try:
        trainer.collect_stats()
    finally:
        logging.getLogger().setLevel(nivel)
    print("  collect_stats OK")


def run_experiment(cambios):
    cfg = {**DEFAULTS, **cambios}
    tag = hacer_tag(cfg)

    if SKIP_IF_DONE and tag in experimentos_hechos():
        print(f"[SALTADO] {tag}\n")
        return None

    print("=" * 78)
    print(f"[EXPERIMENTO] {tag}")
    print("=" * 78)

    exp_dir = f"{OUTPUT_DIR}/{tag}"
    os.makedirs(exp_dir, exist_ok=True)
    drive_dir = f"{RESULTS_DIR}/{tag}"
    os.makedirs(drive_dir, exist_ok=True)

    train_raw, val_raw, stats_dir, n_ses = get_datos(cfg)
    os.makedirs(stats_dir, exist_ok=True)

    data_info = make_data_info(cfg)
    train_ds = ez.dataset.ESPnetEZDataset(train_raw, data_info=data_info)
    valid_ds = ez.dataset.ESPnetEZDataset(val_raw,   data_info=data_info)

    set_encoder_lr_scale(cfg["enc_lr_scale"])
    log_path = f"{exp_dir}/train.log"

    def entrenar(batch_size):
        """Un intento de entrenamiento con el batch dado."""
        c = {**cfg, "batch_size": batch_size}
        ft = build_finetune_config(c, stats_dir)
        pasos = max(1, len(train_raw) // max(1, batch_size))
        print(f"  batch={batch_size} · ~{pasos} pasos/epoca · warmup {c['warmup']} "
              f"(~{c['warmup']/pasos:.1f} epocas)")
        trainer = ez.Trainer(
            task="s2t", train_config=ft,
            train_dataset=train_ds, valid_dataset=valid_ds,
            build_model_fn=make_build_model_fn(c, n_ses), data_info=data_info,
            output_dir=exp_dir, stats_dir=stats_dir, ngpu=NGPU,
        )
        asegurar_stats(trainer, stats_dir)
        train_raw.cerrar(); val_raw.cerrar()   # antes del fork (HDF5 no es fork-safe)
        with CapturaLog(log_path):
            trainer.train()
        del trainer

    t0 = time.time()
    error, bs_usado = "", cfg["batch_size"]
    for intento, bs in enumerate([cfg["batch_size"], max(1, cfg["batch_size"] // 2),
                                  max(1, cfg["batch_size"] // 4)]):
        try:
            bs_usado = bs
            entrenar(bs)
            error = ""
            break
        except torch.cuda.OutOfMemoryError as e:
            gc.collect(); torch.cuda.empty_cache()
            error = f"OutOfMemoryError (batch={bs})"
            if intento < 2:
                print(f"  [OOM con batch={bs}] reintentando con batch={max(1, bs // 2)}...")
            else:
                print(f"  [ERROR] OOM incluso con batch={bs}. Usa una GPU con mas VRAM.")
        except KeyboardInterrupt:
            error = "interrumpido"
            print("  [interrumpido por el usuario]")
            break
        except Exception as e:
            error = f"{type(e).__name__}: {e}"
            print(f"  [ERROR] {error}")
            break
    minutos = (time.time() - t0) / 60

    # ── metricas por epoca ──
    df_ep = parse_train_log(log_path)
    mejores = {}
    if len(df_ep):
        df_ep.insert(0, "exp_tag", tag)
        df_ep.to_csv(f"{drive_dir}/epocas.csv", index=False)
        mejores["epocas_hechas"] = int(df_ep["epoch"].max())
        # El minimo de cer_ctc SOBRE TODAS LAS EPOCAS. Es lo que hay que mirar: el
        # cer_ctc "en la epoca de menor loss" puede ser mucho peor si la loss esta
        # dominada por la perdida de atencion.
        if "valid_cer_ctc" in df_ep.columns and df_ep["valid_cer_ctc"].notna().any():
            j = df_ep["valid_cer_ctc"].idxmin()
            mejores["min_valid_cer_ctc"] = float(df_ep.loc[j, "valid_cer_ctc"])
            mejores["epoch_min_cer_ctc"] = int(df_ep.loc[j, "epoch"])
            ult = df_ep.tail(10)
            if len(ult) >= 5:   # ¿sigue bajando o esta plano?
                mejores["pendiente_cer_ctc"] = float(
                    np.polyfit(ult["epoch"], ult["valid_cer_ctc"], 1)[0])
        if "valid_loss" in df_ep.columns and df_ep["valid_loss"].notna().any():
            mejor = df_ep.loc[df_ep["valid_loss"].idxmin()]
            for k in df_ep.columns:
                if k.startswith("valid_") and pd.notna(mejor[k]):
                    mejores[f"best_{k}"] = float(mejor[k])
            mejores["best_epoch"] = int(mejor["epoch"])

    # ── CER/WER: greedy CTC (honesto) y beam search (con decoder) ──
    cer_m = wer_m = cer_g = wer_g = cer_micro = float("nan")
    if not error or "interrumpido" in error:
        modelo = None
        try:
            modelo = cargar_modelo_entrenado(cfg, exp_dir, n_ses)

            if EVAL_GREEDY_CTC and modelo.ctc is not None:
                print("  evaluando con CTC greedy...")
                cer_micro, cer_g, wer_g = evaluar_ctc_greedy(cfg, modelo, val_raw)
                print(f"  [CTC greedy] cer_ctc(micro, comparable con el log)={cer_micro:.3f} · "
                      f"CER texto={cer_g:.3f} · WER texto={wer_g:.3f}")

            if getattr(modelo, "decoder", None) is not None:
                print("  evaluando con beam search...")
                cer_m, wer_m, n_ev = evaluar(cfg, modelo, val_raw)
                print(f"  [beam]       CER={cer_m:.3f} · WER={wer_m:.3f} ({n_ev} trials)")
            else:
                print("  (sin decoder: ctc_weight=1, se omite el beam search)")
        except Exception as e:
            print(f"  [evaluacion fallida] {type(e).__name__}: {e}")
        finally:
            del modelo
            gc.collect(); torch.cuda.empty_cache()

    # ── guardado ──
    for artefacto in ["train.log", "config.yaml"]:
        origen = f"{exp_dir}/{artefacto}"
        if os.path.exists(origen):
            shutil.copy(origen, f"{drive_dir}/{artefacto}")
    ckpt_tmp = f"{exp_dir}/checkpoint.pth"          # modelo+optimizador, ~1.4 GB
    if GUARDAR_CKPT:
        try:
            shutil.copy(find_best_checkpoint(exp_dir), f"{drive_dir}/valid.loss.best.pth")
        except Exception as e:
            print("  [aviso] no se pudo copiar el checkpoint:", e)
    if os.path.exists(ckpt_tmp) and not error:
        os.remove(ckpt_tmp)                          # libera disco de la VM

    fila = {"exp_tag": tag, "gpu": GPU_NAME, "minutos": round(minutos, 1),
            "compute_units": round(CU_H * minutos / 60, 2),
            "n_train": len(train_raw), "n_val": len(val_raw), "n_sesiones": n_ses,
            "batch_usado": bs_usado,
            "cer_ctc_micro": cer_micro,
            "cer_ctc_greedy": cer_g, "wer_ctc_greedy": wer_g,
            "cer_beam": cer_m, "wer_beam": wer_m, "error": error,
            **{k: v for k, v in cfg.items()}, **mejores}
    guardar_resultado(fila)

    del train_ds, valid_ds
    gc.collect(); torch.cuda.empty_cache()

    print(f"  hecho en {minutos:.1f} min (~{CU_H*minutos/60:.2f} compute units)\n")
    return fila

print("run_experiment listo.")

run_experiment listo.


## 11. El barrido

Cada entrada es solo lo que **cambia** respecto a `DEFAULTS`. El barrido está por fases: **lanza la fase 1 sola** y mira el resultado antes de seguir.

### Por qué el diagnóstico va primero

Lo que dijo tu primera tanda, en una línea: `acc` empieza en 0.557 y acaba en 0.578 tras 30 épocas. Esa `acc` es la del decoder **con teacher forcing** — se le dan los tokens correctos anteriores y se mide si acierta el siguiente. Un ~57 % es lo que saca OWSM usando solo estadística del inglés, sin mirar la entrada. Que arranque ahí y no se mueva significa que el encoder no aporta nada. Y `wer=1.000` con `cer=0.34` lo confirma desde el otro lado: escribe inglés bien formado y no acierta ni una palabra.

El `overfit` de la fase 1 elimina todas las salidas de escape a la vez: una sesión (sin deriva entre días), val = train (sin generalización que valga), `ctc_weight=1.0` (ESPnet desactiva el decoder, así que no hay prior lingüístico donde esconderse) y 200 épocas. Lo que quede es la capacidad del encoder de mapear sEEG → fonemas. Son ~10 min y 0.3 unidades.

### Por qué la capa por sesión

Es *la* característica de este dataset: 45 sesiones repartidas en 20 meses, y la señal que produce un mismo fonema cambia de un día a otro por deriva de los electrodos. El algoritmo de referencia de B2T'25 pasa las características por una capa lineal específica del día seguida de un `softsign` antes de nada más, y los trabajos cross-subject recientes usan transformadas afines por día para alinear todo a un espacio común.

Tú tenías un único frontend compartido para 10 sesiones: el modelo recibía diez mapeos señal→fonema contradictorios y tenía que deducir de qué día venía cada trial. `CapaSesion` aprende una matriz 512×512 más un sesgo por sesión, inicializados a la identidad, y aplica `softsign` después. El índice de sesión viaja hasta el frontend como un canal extra del tensor de entrada (512 → 513).

### Otros cambios respecto a la tanda anterior

- **`enc_lr_scale=0.01`** en los dos runs de la fase 2: el encoder preentrenado a 1e-5 y el frontend nuevo a 1e-3. Con todo a 1e-3 lo más probable es que el warmup estuviera destruyendo los pesos de OWSM.
- **`cer_ctc_greedy`** es la métrica a mirar en la tabla, no `cer_beam`. Sale de decodificar el `argmax` de la cabeza CTC colapsando repeticiones: sin decoder, sin beam, sin prior. Es la que no se puede falsear.

In [15]:
BARRIDO = [
    # ══ EL EXPERIMENTO DECISIVO — lanza SOLO este ═══════════════════════════
    # Identico al `overfit` que llego a cer_ctc=0.117, cambiando UNA variable:
    # val es ahora un split real en vez de val=train. Misma sesion, mismo
    # ctc_weight=1.0, mismas 200 epocas. Aisla memorizacion vs generalizacion.
    #   ~1.4 h · ~2.4 unidades
    dict(nombre="gen1", n_sessions=1, max_epoch=200, warmup=50,
         val_igual_train=False, frontend="linear", ctc_weight=1.0),

    # ══ RAMA A · si gen1 baja de ~0.5 (generaliza dentro de un dia) ═════════
    # El problema seria multi-sesion -> la capa por sesion merece otra
    # oportunidad, ahora sobre una configuracion que si aprende.
    # dict(nombre="A_sin", n_sessions=10, max_epoch=120, warmup=200,
    #      frontend="linear", ctc_weight=1.0, capa_sesion=False),
    # dict(nombre="A_con", n_sessions=10, max_epoch=120, warmup=200,
    #      frontend="linear", ctc_weight=1.0, capa_sesion=True),

    # ══ RAMA B · si gen1 se queda en ~0.9 (no generaliza ni con un dia) ═════
    # Entonces es volumen de datos y las 50.002 clases. Primero lo barato:
    # 4,5x mas datos (45 sesiones = ~10.300 frases) y CTC puro.
    #   ~4,5 h · ~8 unidades
    # dict(nombre="B_full", n_sessions=45, max_epoch=40, warmup=500,
    #      frontend="linear", ctc_weight=1.0, capa_sesion=True, batch_size=32),

    # ══ CONTROLES BARATOS (cualquiera de las dos ramas) ═════════════════════
    # Solo LoRA + frontend: 1,3 M entrenables en vez de 45 M. Ataca directamente
    # el sobreajuste (best_epoch=4 de 30 en toda la tanda anterior).
    # dict(nombre="C_frozen", n_sessions=10, max_epoch=120, warmup=200,
    #      frontend="linear", ctc_weight=1.0, unfreeze_encoder=False),
    # Sin re-normalizar: los HDF5 ya vienen z-scoreados por bloque.
    # dict(nombre="C_norm", n_sessions=10, max_epoch=120, warmup=200,
    #      frontend="linear", ctc_weight=1.0, norm="ninguna"),
]

# Referencia medida en L4: n=10, 30 epocas, frontend linear -> 59 min (2 min/epoca).
def estimar_min(cfg):
    c = {**DEFAULTS, **cfg}
    factor = 1.0 if "L4" in GPU_NAME else (2.0 if "T4" in GPU_NAME else 0.5)
    por_epoca = 2.0 * (c["n_sessions"] / 10) * (0.8 if c["ctc_weight"] == 1.0 else 1.0)
    return c["max_epoch"] * por_epoca * factor

total_min = sum(estimar_min(e) for e in BARRIDO)
print(f"GPU: {GPU_NAME} ({CU_H} unidades/h)")
print(f"{len(BARRIDO)} experimentos · ~{total_min/60:.1f} h · "
      f"~{CU_H*total_min/60:.1f} compute units (~${CU_H*total_min/60*0.10:.2f})\n")
for e in BARRIDO:
    print(f"  ~{estimar_min(e)/60:4.1f} h · {hacer_tag({**DEFAULTS, **e})}")

GPU: NVIDIA L4 (1.71 unidades/h)
1 experimentos · ~0.5 h · ~0.9 compute units (~$0.09)

  ~ 0.5 h · gen1_n1_linear_unf1_lora1_lr0.001_els1_ctc1_bs16_ep200


## 12. Ejecutar

Con Colab Pro+ puedes activar la ejecución en segundo plano y cerrar el navegador. Si te desconectan a mitad, vuelve a lanzar esta celda: los experimentos terminados se saltan y el que estuviera a medias reanuda desde su `checkpoint.pth`.

In [16]:
inicio = time.time()
for cambios in BARRIDO:
    run_experiment(cambios)

print("=" * 78)
print(f"BARRIDO COMPLETO en {(time.time()-inicio)/60:.0f} min "
      f"(~{CU_H*(time.time()-inicio)/3600:.1f} compute units)")
print("Resultados en:", CSV_RESULTADOS)

[EXPERIMENTO] gen1_n1_linear_unf1_lora1_lr0.001_els1_ctc1_bs16_ep200
Sesiones (1): ['t15.2023.08.13']
Train: 348 trials · Val: 35 trials
  batch=16 · ~21 pasos/epoca · warmup 50 (~2.4 epocas)
  recopilando estadisticas (solo la primera vez por configuracion de datos)...


/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-9192b124-ac49-49a6-afd7-ff66f573d81c.json


  frontend=linear · lora=True · encoder=descongelado · ctc_weight=1.0 · norm=trial
  45,165,778 entrenables / 45,165,778 totales (100.00%)
  [nota] ctc_weight=1 -> ESPnet desactiva el decoder (solo CTC; la evaluacion sera greedy)
  collect_stats OK


/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-9192b124-ac49-49a6-afd7-ff66f573d81c.json
    Saving the configuration in /content/exp/gen1_n1_linear_unf1_lora1_lr0.001_els1_ctc1_bs16_ep200/config.yaml


  frontend=linear · lora=True · encoder=descongelado · ctc_weight=1.0 · norm=trial
  45,165,778 entrenables / 45,165,778 totales (100.00%)
  [nota] ctc_weight=1 -> ESPnet desactiva el decoder (solo CTC; la evaluacion sera greedy)


/usr/local/lib/python3.12/dist-packages/espnet2/train/trainer.py:219: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
    1/200epoch started
/usr/local/lib/python3.12/dist-packages/espnet2/train/trainer.py:638: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(
/usr/local/lib/python3.12/dist-packages/espnet2/s2t/espnet_model.py:338: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(False):
    1epoch:train:1-10batch: iter_time=0.307, forward_time=0.130, loss_ctc=44.532, loss=44.532, backward_time=0.107, grad_norm=388.458, clip=100.000, loss_scale=6.554e+04, optim_step_time=0.020, optim0_lr0=1.300e-04train_time=0.586
    1epoch:train:11-20batch: iter_time=0.374, forward_time=0.059, loss_ctc=48.020, loss=48.020,

  evaluando con CTC greedy...


/usr/local/lib/python3.12/dist-packages/espnet2/s2t/espnet_model.py:338: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(False):


    [ctc] REF: you can see the code at this point as well
    [ctc] HYP: he stay them
    [ctc] REF: how does it keep the cost down
    [ctc] HYP: i not state
    [ctc] REF: not too controversial
    [ctc] HYP: family not guilty
    por trial: mediana=0.800 p10=0.725 p90=0.859 · 0% por debajo de 0.2
  [CTC greedy] cer_ctc(micro, comparable con el log)=0.769 · CER texto=0.795 · WER texto=0.970
  (sin decoder: ctc_weight=1, se omite el beam search)
  hecho en 51.0 min (~1.45 compute units)

BARRIDO COMPLETO en 51 min (~1.5 compute units)
Resultados en: /content/drive/MyDrive/TFG/resultados/resultados.csv


## 13. Resultados y curvas

Tabla comparativa para la memoria y curvas de entrenamiento de cada experimento.

In [17]:
import matplotlib.pyplot as plt

df = pd.read_csv(CSV_RESULTADOS)
cols = ["nombre", "frontend", "capa_sesion", "norm", "unfreeze_encoder", "lr", "enc_lr_scale",
        "ctc_weight", "n_train", "best_epoch", "best_valid_loss", "best_valid_acc",
        "best_valid_cer_ctc", "cer_ctc_greedy", "wer_ctc_greedy", "cer_beam", "wer_beam",
        "minutos", "compute_units"]
cols = [c for c in cols if c in df.columns]
resumen = df[cols].sort_values("best_valid_loss" if "best_valid_loss" in df else "nombre")
display(resumen)

resumen.to_csv(f"{RESULTS_DIR}/tabla_resumen.csv", index=False)
print("Tabla guardada en", f"{RESULTS_DIR}/tabla_resumen.csv")
print("\nLaTeX para la memoria:\n")
print(resumen.to_latex(index=False, float_format="%.3f"))

,nombre,frontend,capa_sesion,norm,unfreeze_encoder,lr,enc_lr_scale,ctc_weight,n_train,best_epoch,best_valid_loss,best_valid_acc,best_valid_cer_ctc,cer_ctc_greedy,wer_ctc_greedy,cer_beam,wer_beam,minutos,compute_units
8,overfit,linear,False,trial,True,0.001,1.00,1.0,348,199.0,1.724,NaN,0.117,0.581225,0.684772,NaN,NaN,82.5,2.35
0,sanity,linear,NaN,NaN,True,0.001,1.00,0.3,545,14.0,28.731,0.605,0.868,NaN,NaN,0.749266,1.056681,17.3,0.34
3,linear,linear,NaN,NaN,True,0.001,1.00,0.3,2296,5.0,36.759,0.564,0.924,NaN,NaN,0.795415,1.069532,101.7,2.02
9,sinSesion,linear,False,trial,True,0.001,0.01,0.3,2296,4.0,36.913,0.559,0.952,0.986814,0.987778,0.862936,1.397937,59.2,1.69
10,conSesion,linear,True,trial,True,0.001,0.01,0.3,2296,5.0,37.102,0.553,0.989,0.995862,0.997778,0.796121,1.128944,59.4,1.69
4,conv1d,conv1d,NaN,NaN,True,0.001,1.00,0.3,2296,1.0,37.523,0.558,1.000,NaN,NaN,0.842685,1.235167,104.0,2.06
11,gen1,linear,False,trial,True,0.001,1.00,1.0,348,1.0,48.928,NaN,1.000,0.795170,0.969841,NaN,NaN,51.0,1.45
1,base,conv2d,NaN,NaN,True,0.001,1.00,0.3,2296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.1,0.00
2,lrdif,conv2d,NaN,NaN,True,0.001,0.01,0.3,2296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.1,0.00
5,frozen,conv2d,NaN,NaN,False,0.001,1.00,0.3,2296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.1,0.00


Tabla guardada en /content/drive/MyDrive/TFG/resultados/tabla_resumen.csv

LaTeX para la memoria:

\begin{tabular}{llllrrrrrrrrrrrrrrr}
\toprule
nombre & frontend & capa_sesion & norm & unfreeze_encoder & lr & enc_lr_scale & ctc_weight & n_train & best_epoch & best_valid_loss & best_valid_acc & best_valid_cer_ctc & cer_ctc_greedy & wer_ctc_greedy & cer_beam & wer_beam & minutos & compute_units \\
\midrule
overfit & linear & False & trial & True & 0.001 & 1.000 & 1.000 & 348 & 199.000 & 1.724 & NaN & 0.117 & 0.581 & 0.685 & NaN & NaN & 82.500 & 2.350 \\
sanity & linear & NaN & NaN & True & 0.001 & 1.000 & 0.300 & 545 & 14.000 & 28.731 & 0.605 & 0.868 & NaN & NaN & 0.749 & 1.057 & 17.300 & 0.340 \\
linear & linear & NaN & NaN & True & 0.001 & 1.000 & 0.300 & 2296 & 5.000 & 36.759 & 0.564 & 0.924 & NaN & NaN & 0.795 & 1.070 & 101.700 & 2.020 \\
sinSesion & linear & False & trial & True & 0.001 & 0.010 & 0.300 & 2296 & 4.000 & 36.913 & 0.559 & 0.952 & 0.987 & 0.988 & 0.863 & 1.398 & 59.200

In [18]:
# Curvas de todos los experimentos con epocas.csv
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for tag in df["exp_tag"]:
    p = f"{RESULTS_DIR}/{tag}/epocas.csv"
    if not os.path.exists(p):
        continue
    e = pd.read_csv(p)
    etiqueta = tag.split("_")[0]
    if "train_loss" in e: axes[0].plot(e["epoch"], e["train_loss"], label=etiqueta)
    if "valid_loss" in e: axes[1].plot(e["epoch"], e["valid_loss"], label=etiqueta)
    if "valid_cer"  in e: axes[2].plot(e["epoch"], e["valid_cer"],  label=etiqueta)

for ax, t in zip(axes, ["Loss (train)", "Loss (valid)", "CER (valid)"]):
    ax.set_title(t); ax.set_xlabel("epoca"); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/curvas.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en", f"{RESULTS_DIR}/curvas.png")

Figura guardada en /content/drive/MyDrive/TFG/resultados/curvas.png


---

## Cómo leer `gen1`

Es el mismo experimento que el `overfit` (que llegó a `cer_ctc` = 0.117) con **una sola** variable cambiada: la validación es ahora un split real en vez de `val = train`.

| `min_valid_cer_ctc` | Lectura | Siguiente paso |
|---|---|---|
| < 0.5 | Generaliza dentro de un día. El cuello de botella es multi-sesión. | Rama A: la ablación de la capa por sesión, ya sobre algo que aprende. |
| 0.7 – 0.9 | Generaliza un poco. Falta señal o faltan datos. | Rama B: las 45 sesiones. |
| ≈ 0.9 y `pendiente_cer_ctc` ≈ 0 | No generaliza ni con un día, sin deriva ni confusión entre sesiones. | Rama B + cambiar la cabeza CTC (ver abajo). |

Mira siempre `pendiente_cer_ctc` (ajuste lineal sobre las últimas 10 épocas). En `sinSesion` valía −0.0008/época — plano. En el `overfit` a la misma altura valía −0.014/época — despegando. Es lo que distingue "va lento" de "está atascado".

## El problema de las 50.002 clases

Si `gen1` sale plano, este es el cambio estructural que toca. Los pipelines de referencia de B2T'25 no predicen BPE: su capa final tiene **41 clases** (39 fonemas + blank + espacio), y las palabras salen después con un n-grama o un LLM. Tú le estás pidiendo a la cabeza CTC que aprenda un mapeo neuronal → 50.002 tokens con 2.296 frases.

Dos versiones, de menos a más trabajo:

1. **Caracteres (~30 clases).** No hace falta ninguna dependencia nueva: tokenizas `text_ctc` letra a letra y sustituyes `model.ctc.ctc_lo` por un `nn.Linear(384, 30)`. Captura casi toda la reducción de clases.
2. **Fonemas (41 clases).** Lo que hace la literatura, con `g2p_en`. Comparable con los números publicados del benchmark, que es lo que quieres para la memoria.

En los dos casos `cer_ctc` deja de ser comparable con las tablas anteriores (pasa a ser CER de caracteres o PER), así que conviene documentarlo como una fase aparte.

## Ajustes de rendimiento

- Si la GPU va a menos del 70 % (mira `!nvidia-smi` desde el Terminal), sube `batch_size` o pasa a `batch_type="numel"` con `batch_bins` (agrupa por número de elementos, mucho mejor con longitudes de 138 a 1966 frames).
- `num_workers=0` es lo seguro con HDF5. Súbelo solo si ves la GPU esperando.
- El reintento automático por OOM baja el batch a la mitad y luego a un cuarto antes de rendirse, así que un experimento mal dimensionado ya no tumba el barrido entero.